In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
# import pearsonr
from scipy.stats import pearsonr
from utils.utils import load_encrypted_xlsx

In [ ]:
registry_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
pdms_link_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv'
outcome_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'
nor_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_EinzelGabeNoradrSpritzenpumpe.csv'
all_drugs_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20250401_Medikamente.csv'

In [ ]:
registry_df = load_encrypted_xlsx(registry_data_path)
outcome_df = load_encrypted_xlsx(outcome_path)
registry_pdms_correspondance_df = pd.read_csv(pdms_link_path)

nor_df = pd.read_csv(nor_data_path, sep= ';', decimal='.')
all_drugs_df = pd.read_csv(all_drugs_data_path, sep=';', decimal='.', header=None)

In [ ]:
restrict_to_d2_d21 = True

In [ ]:
nor_df = pd.read_csv(nor_data_path, sep= ';', decimal='.')

all_drugs_df = pd.read_csv(all_drugs_data_path, sep=';', decimal='.', header=None)
all_drugs_df.columns = ['pNr', 'drugID', 'drugName', 'Start', 'Ende', 'Dauer', 'Menge', 'not_clear', 'Einheit']

In [ ]:
registry_df.drop_duplicates(inplace=True)
registry_df.dropna(subset=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], inplace=True)

In [ ]:
registry_pdms_correspondance_df.rename(columns={
    'JoinedName': 'Name',
}, inplace=True)
registry_pdms_correspondance_df['Date_birth']=pd.to_datetime(registry_pdms_correspondance_df['Date_birth'], format='%d.%m.%Y')
registry_df['Date_birth'] = pd.to_datetime(registry_df['Date_birth'], format='%d.%m.%Y')
registry_df = registry_df.merge(registry_pdms_correspondance_df, how='left', on=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'])
outcome_df['Date_birth'] = pd.to_datetime(outcome_df['Date_birth'])
outcome_df = outcome_df.merge(registry_pdms_correspondance_df, how='left', on=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'])


In [ ]:
# Preprocess noradrenaline data
# only accept noradrenaline data with Einheit 'MICROGRAM' and 'MILLIGRAM'
nor_df = nor_df[nor_df['Einheit'].isin(['MICROGRAM', 'MILLIGRAM'])]
# convert Menge to MICROGRAM where Einheit is MILLIGRAM
nor_df.loc[nor_df['Einheit'] == 'MILLIGRAM', 'Menge'] = nor_df['Menge'] * 1000
nor_df = nor_df.drop(columns=['Einheit'])

# filter out rows where Menge or Dauer is 0 or NaN
nor_df = nor_df[(nor_df['Menge'] != 0) & (nor_df['Dauer'] != 0)]
nor_df = nor_df.dropna(subset=['Menge', 'Dauer'])


In [ ]:
nimodipine_name_list = ['Nimotop', 'Nimodipin']
nimodipine_df = all_drugs_df[all_drugs_df['drugName'].isin(nimodipine_name_list)]
# only consider PO (per os) administration (this adminsitration length is 1)
nimodipine_df = nimodipine_df[nimodipine_df.Dauer == 1]

In [ ]:
nor_df.drop_duplicates(inplace=True)
nimodipine_df.drop_duplicates(inplace=True)

# Transform to relative timing
- T0 = date ictus or admission or first nimo admin

In [ ]:
registry_df.Date_Ictus.fillna(registry_df.Date_admission, inplace=True)
registry_df['Date_Ictus'] = pd.to_datetime(registry_df['Date_Ictus'], format='%d.%m.%Y')
registry_df['T0'] = registry_df['Date_Ictus'] 

first_nimodipine_administration = nimodipine_df.groupby('pNr').agg({
    'Start': 'min',
}).reset_index().rename(columns={'Start': 'first_nimodipine_administration'})

registry_df = registry_df.merge(first_nimodipine_administration, how='left', on='pNr')
# Convert first_nimodipine_administration to datetime
registry_df['first_nimodipine_administration'] = pd.to_datetime(registry_df['first_nimodipine_administration'])
registry_df['T0'].fillna(registry_df['first_nimodipine_administration'], inplace=True)

In [ ]:
nor_df = nor_df.merge(
    registry_df[['pNr', 'T0']],
    how='left',
    on='pNr')
nor_df['relative_start_h'] = (pd.to_datetime(nor_df['Start']) - nor_df['T0']).dt.total_seconds() / 3600
nor_df['relative_end_h'] = (pd.to_datetime(nor_df['Ende']) - nor_df['T0']).dt.total_seconds() / 3600
nor_df['relative_start_d'] = nor_df['relative_start_h'] / 24
nor_df['relative_end_d'] = nor_df['relative_end_h'] / 24
# Create categorical variables for relative start and end times (flooring the values)
nor_df['relative_start_d_cat'] = nor_df['relative_start_d'].apply(lambda x: np.floor(x))
nor_df['relative_end_d_cat'] = nor_df['relative_end_d'].apply(lambda x: np.floor(x))

daily_nor_df = nor_df.groupby(['pNr', 'relative_start_d_cat']).agg({
    'Menge': 'sum',
}).reset_index()


In [ ]:
nimodipine_df = nimodipine_df.merge(
    registry_df[['pNr', 'T0']],
    how='left',
    on='pNr')
nimodipine_df['relative_start_h'] = (pd.to_datetime(nimodipine_df['Start']) - nimodipine_df['T0']).dt.total_seconds() / 3600
nimodipine_df['relative_start_d'] = nimodipine_df['relative_start_h'] / 24
nimodipine_df['relative_start_d_cat'] = nimodipine_df['relative_start_d'].apply(lambda x: np.floor(x))
nimodipine_daily_df = nimodipine_df.groupby(['pNr', 'relative_start_d_cat']).agg({
    'Menge': 'sum',
}).reset_index()

In [ ]:
if restrict_to_d2_d21:
    daily_nor_df = daily_nor_df[(daily_nor_df['relative_start_d_cat'] >= 2) & (daily_nor_df['relative_start_d_cat'] <= 21)]
    nimodipine_daily_df = nimodipine_daily_df[(nimodipine_daily_df['relative_start_d_cat'] >= 2) & (nimodipine_daily_df['relative_start_d_cat'] <= 21)]

    nor_df = nor_df[(nor_df['relative_start_d_cat'] >= 2) & (nor_df['relative_start_d_cat'] <= 21)]
    nimodipine_df = nimodipine_df[(nimodipine_df['relative_start_d_cat'] >= 2) & (nimodipine_df['relative_start_d_cat'] <= 21)]

In [ ]:
# add 4h intervals dating
nimodipine_df['relative_start_4h_cat'] = (nimodipine_df['relative_start_h'] / 4).apply(lambda x: np.floor(x))
nor_df['relative_start_4h_cat'] = (nor_df['relative_start_h'] / 4).apply(lambda x: np.floor(x))
nor_df['relative_start_4h'] = (nor_df['relative_start_h'] / 4)
nor_df['relative_end_4h'] = (nor_df['relative_end_h'] / 4)

In [ ]:
def split_administrations_by_4h_intervals(nor_df):
    """
    Split noradrenaline administrations that span multiple 4-hour intervals into 
    separate entries for each 4-hour interval they span.
    
    Parameters:
    -----------
    nor_df : DataFrame
        DataFrame containing noradrenaline administrations with 'relative_start_4h', 
        'relative_end_4h', 'Menge', 'Dauer' columns
    
    Returns:
    --------
    DataFrame
        Expanded DataFrame where administrations are split by 4h intervals
    """
    split_rows = []
    
    for _, row in nor_df.iterrows():
        if np.isnan(row['relative_start_4h']) or np.isnan(row['relative_end_4h']):
            # If either start or end is None, skip this row
            split_rows.append(row.to_dict())
            continue
        
        # Calculate the start and end 4h intervals (as integers)
        start_interval = int(np.floor(row['relative_start_4h']))
        end_interval = int(np.floor(row['relative_end_4h']))
        
        # If administration is within the same 4h interval, keep as is
        if start_interval == end_interval:
            split_rows.append(row.to_dict())
            continue
        
        # For administrations spanning multiple intervals
        total_duration = row['Dauer']  # Original duration in minutes
        total_amount = row['Menge']    # Original amount
        rate = total_amount / total_duration  # Amount per minute
        
        # Process each interval the administration spans
        for interval in range(start_interval, end_interval + 1):
            # Create a copy of the original row
            new_row = row.to_dict()
            
            # Set the 4h interval category
            new_row['relative_start_4h_cat'] = interval
            
            # Calculate start and end times within this interval
            if interval == start_interval:
                # First interval: from original start to end of interval
                interval_start = row['relative_start_4h'] * 4  # Convert back to hours
                interval_end = (interval + 1) * 4  # End of interval in hours
                new_row['relative_start_h'] = interval_start
                new_row['relative_end_h'] = interval_end
            elif interval == end_interval:
                # Last interval: from start of interval to original end
                interval_start = interval * 4  # Start of interval in hours
                interval_end = row['relative_end_4h'] * 4  # Convert back to hours
                new_row['relative_start_h'] = interval_start
                new_row['relative_end_h'] = interval_end
            else:
                # Middle intervals: full 4-hour durations
                new_row['relative_start_h'] = interval * 4
                new_row['relative_end_h'] = (interval + 1) * 4
            
            # Calculate duration for this interval (in minutes)
            interval_duration_hours = new_row['relative_end_h'] - new_row['relative_start_h']
            interval_duration_minutes = interval_duration_hours * 60
            new_row['Dauer'] = interval_duration_minutes
            
            # Calculate amount for this interval based on rate
            new_row['Menge'] = rate * interval_duration_minutes

            new_row['rate'] = rate  # Add rate for reference
            
            # Update start and end dates
            time_delta_start = pd.Timedelta(hours=new_row['relative_start_h'])
            time_delta_end = pd.Timedelta(hours=new_row['relative_end_h'])
            new_row['Start'] = row['T0'] + time_delta_start
            new_row['Ende'] = row['T0'] + time_delta_end
            
            # Add to results
            split_rows.append(new_row)
    
    # Convert list of dictionaries back to DataFrame
    result_df = pd.DataFrame(split_rows)
    
    # Recalculate 4h categories to ensure consistency
    result_df['relative_start_4h'] = result_df['relative_start_h'] / 4
    result_df['relative_end_4h'] = result_df['relative_end_h'] / 4
    result_df['relative_start_4h_cat'] = result_df['relative_start_4h'].apply(lambda x: np.floor(x))
    result_df['relative_end_4h_cat'] = result_df['relative_end_4h'].apply(lambda x: np.floor(x))
    
    # Also update daily categories
    result_df['relative_start_d'] = result_df['relative_start_h'] / 24
    result_df['relative_end_d'] = result_df['relative_end_h'] / 24
    result_df['relative_start_d_cat'] = result_df['relative_start_d'].apply(lambda x: np.floor(x))
    result_df['relative_end_d_cat'] = result_df['relative_end_d'].apply(lambda x: np.floor(x))
    
    return result_df

In [ ]:
nor_4h_cat_df = split_administrations_by_4h_intervals(nor_df)
nor_4h_cat_df = nor_4h_cat_df.groupby(['pNr', 'relative_start_4h_cat']).agg({
    'Menge': 'sum',
}).reset_index()
nimodipine_4h_cat_df = nimodipine_df.groupby(['pNr', 'relative_start_4h_cat']).agg({
    'Menge': 'sum',
}).reset_index()

In [ ]:
nimo_nor_daily_df = daily_nor_df.merge(
    nimodipine_daily_df,
    how='outer',
    on=['pNr', 'relative_start_d_cat'],
    suffixes=('_nor', '_nimo')
)
nimo_nor_4h_cat_df = nor_4h_cat_df.merge(
    nimodipine_4h_cat_df,
    how='outer',
    on=['pNr', 'relative_start_4h_cat'],
    suffixes=('_nor', '_nimo')
)

In [ ]:
nimo_nor_daily_df['Menge_nimo'] = nimo_nor_daily_df['Menge_nimo'].fillna(0)
nimo_nor_daily_df['Menge_nor'] = nimo_nor_daily_df['Menge_nor'].fillna(0)

nimo_nor_4h_cat_df['Menge_nimo'] = nimo_nor_4h_cat_df['Menge_nimo'].fillna(0)
nimo_nor_4h_cat_df['Menge_nor'] = nimo_nor_4h_cat_df['Menge_nor'].fillna(0)

In [ ]:
outcome_df.pNr.isna().sum(), outcome_df.pNr.nunique()

In [ ]:
nimo_nor_daily_df = nimo_nor_daily_df.merge(
    registry_df[['pNr', 'T0']],
    how='left',
    on='pNr'
)

In [ ]:
# if mRS_discharge 6 ,let mRS_FU_1y = 6
outcome_df['mRS_discharge_6'] = outcome_df['mRS_discharge'].apply(lambda x: 6 if x == 6 else pd.NA)
outcome_df['mRS_FU_1y'].fillna(outcome_df['mRS_discharge_6'], inplace=True)
outcome_df["mRS_FU_1y"]=pd.to_numeric(outcome_df['mRS_FU_1y'], errors='coerce')


In [ ]:
nimo_nor_daily_df = nimo_nor_daily_df.merge(
    outcome_df[['pNr', 'mRS_FU_1y']],
    how='left',
    on='pNr'
)

nimo_nor_daily_df = nimo_nor_daily_df.merge(
    registry_df[['pNr', 'DCI_ischemia']],
    how='left',
    on='pNr'
)

nimo_nor_4h_cat_df = nimo_nor_4h_cat_df.merge(
    outcome_df[['pNr', 'mRS_FU_1y']],
    how='left',
    on='pNr'
)

nimo_nor_4h_cat_df = nimo_nor_4h_cat_df.merge(
    registry_df[['pNr', 'DCI_ischemia']],
    how='left',
    on='pNr'
)

In [ ]:
nimo_nor_4h_cat_df

In [ ]:
nimo_nor_daily_df

In [ ]:
# let function be (nimo ^2)/(nor + 1)
nimo_nor_daily_df['function'] = (nimo_nor_daily_df['Menge_nimo'] ** 1) / (nimo_nor_daily_df['Menge_nor'] + 1)

In [ ]:
nimo_nor_daily_df[['Menge_nimo', 'Menge_nor']].shape

In [ ]:
# boxplot of Menge_nor and Menge_nimo by relative_start_d_cat 
sns.set(style="whitegrid")
plt.figure(figsize=(12, 6))

plot_function = True

ax = sns.lineplot(data=nimo_nor_daily_df, x='relative_start_d_cat', y='Menge_nimo', color='blue', label='Nimodipine')
ax2 = ax.twinx()
sns.lineplot(data=nimo_nor_daily_df, x='relative_start_d_cat', y='Menge_nor', color='orange', ax=ax2, label='Noradrenaline')
ax2.grid(False)

ax.set_xlabel('Relative Start Day (d)')
ax.set_ylabel('Nimodipine Daily Dose (mg)')
ax2.set_ylabel('Noradrenaline Daily Dose (micrograms)')

if plot_function:
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))  # move the third y-axis outward
    sns.lineplot(data=nimo_nor_daily_df, x='relative_start_d_cat', y='function', color='green', ax=ax3, label='Function')
    ax3.set_ylabel('Function')
    ax3.grid(False)

# redraw the legend
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
# remove original legend
ax.get_legend().remove()
ax2.get_legend().remove()

ax.legend(lines + lines2, labels + labels2, loc='upper left')


In [ ]:
# boxplot of Menge_nor and Menge_nimo by relative_start_4h_cat
plt.figure(figsize=(12, 6))
ax = sns.lineplot(data=nimo_nor_4h_cat_df, x='relative_start_4h_cat', y='Menge_nimo', color='blue', label='Nimodipine')
ax2 = ax.twinx()
sns.lineplot(data=nimo_nor_4h_cat_df, x='relative_start_4h_cat', y='Menge_nor', color='orange', ax=ax2, label='Noradrenaline')
ax2.grid(False)

ax.set_xlabel('Relative Start 4h Category')
ax.set_ylabel('Nimodipine 4h Dose (mg)')
ax2.set_ylabel('Noradrenaline 4h Dose (micrograms)')

# redraw the legend
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
# remove original legend
ax.get_legend().remove()
ax2.get_legend().remove()
ax.legend(lines + lines2, labels + labels2, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
temp_df = nimo_nor_daily_df[nimo_nor_daily_df.pNr == 33882]
# plot nor and nimo over time
plt.figure(figsize=(12, 6))
ax = sns.lineplot(data=temp_df, x='relative_start_d_cat', y='Menge_nimo', marker='x', label='Nimodipine')
# add second y-axis for noradrenaline
ax2 = ax.twinx()
sns.lineplot(data=temp_df, x='relative_start_d_cat', y='Menge_nor', marker='o', ax=ax2, label='Noradrenaline', color='orange')

In [ ]:
# ordinal regression nimo_dose / nor_dose vs mRS_FU_1y
from statsmodels.miscmodels.ordinal_model import OrderedModel
temp_df = nimo_nor_daily_df[['Menge_nimo', 'Menge_nor', 'mRS_FU_1y']]
temp_df["mRS_FU_1y"]=pd.to_numeric(temp_df['mRS_FU_1y'], errors='coerce')
temp_df['Menge_nimo'] = temp_df['Menge_nimo'].astype('float')
temp_df['Menge_nor'] = temp_df['Menge_nor'].astype('float')
temp_df.dropna(subset=['mRS_FU_1y', 'Menge_nimo', 'Menge_nor'], inplace=True)

mod = OrderedModel(
            temp_df['mRS_FU_1y'],
            temp_df[['Menge_nimo', 'Menge_nor']],
            distr='logit'
        )
res = mod.fit(method='bfgs', disp=False)
print(res.summary())

In [ ]:
# ordinal regression nimo_dose / nor_dose vs mRS_FU_1y
from statsmodels.miscmodels.ordinal_model import OrderedModel
temp_df = nimo_nor_daily_df[['function', 'mRS_FU_1y']]
temp_df['function'] = temp_df['function'].astype('float')
temp_df.dropna(subset=['mRS_FU_1y', 'function'], inplace=True)

mod = OrderedModel(
            temp_df['mRS_FU_1y'],
            temp_df[['function']],
            distr='logit'
        )
res = mod.fit(method='bfgs', disp=False)
print(res.summary())

In [ ]:
nimodipine_daily_df.Menge.unique()

In [ ]:
def count_days_above_threshold(df, nimodipine_thresholds, noradrenaline_thresholds, time_category_column='relative_start_d_cat'):
    results = []
    for nimo_thresh in tqdm(nimodipine_thresholds):
        for nor_thresh in noradrenaline_thresholds:
            temp_df = df[(df['Menge_nimo'] >= nimo_thresh) & (df['Menge_nor'] >= nor_thresh)]
            count = temp_df.groupby('pNr').agg({
                time_category_column: 'count',
                'mRS_FU_1y': 'first',
                'DCI_ischemia': 'first'
            }).reset_index().rename(columns={time_category_column: 'days_with_nimo_and_nor_above_threshold'})
            count['nimodipine_threshold'] = nimo_thresh
            count['noradrenaline_threshold'] = nor_thresh
            results.append(count)
    return pd.concat(results)

In [ ]:
def count_days_in_threshold_ranges(df, nimodipine_thresholds, noradrenaline_thresholds, time_category_column='relative_start_d_cat'):
    """
    Count days when both nimodipine and noradrenaline are within specific threshold ranges.
    For each pair of consecutive thresholds, count days when the values are:
    - Above or equal to the lower threshold
    - Below the higher threshold (if not the last threshold)
    
    Parameters:
    -----------
    df : DataFrame
        DataFrame containing 'Menge_nimo', 'Menge_nor', 'pNr', 'relative_start_d_cat', etc.
    nimodipine_thresholds : list
        Sorted list of nimodipine thresholds
    noradrenaline_thresholds : list
        Sorted list of noradrenaline thresholds
    
    Returns:
    --------
    DataFrame
        DataFrame with count of time category in each threshold range per patient
    """
    results = []
    
    for i, nimo_lower in enumerate(nimodipine_thresholds):
        # Determine the upper threshold for nimodipine
        nimo_upper = float('inf') if i == len(nimodipine_thresholds) - 1 else nimodipine_thresholds[i + 1]
        
        for j, nor_lower in enumerate(noradrenaline_thresholds):
            # Determine the upper threshold for noradrenaline
            nor_upper = float('inf') if j == len(noradrenaline_thresholds) - 1 else noradrenaline_thresholds[j + 1]
            
            # Filter data for current threshold range
            temp_df = df[(df['Menge_nimo'] >= nimo_lower) & 
                         (df['Menge_nimo'] < nimo_upper) & 
                         (df['Menge_nor'] >= nor_lower) & 
                         (df['Menge_nor'] < nor_upper)]
            
            # Count days within this range for each patient
            if not temp_df.empty:
                count = temp_df.groupby('pNr').agg({
                    time_category_column: 'count',
                    'mRS_FU_1y': 'first',
                    'DCI_ischemia': 'first'
                }).reset_index()
                
                # Add threshold information
                count['nimodipine_lower'] = nimo_lower
                count['nimodipine_upper'] = nimo_upper if nimo_upper != float('inf') else "max"
                count['noradrenaline_lower'] = nor_lower
                count['noradrenaline_upper'] = nor_upper if nor_upper != float('inf') else "max"
                count['threshold_range'] = f"Nimo: [{nimo_lower}-{nimo_upper if nimo_upper != float('inf') else 'max'}), Nor: [{nor_lower}-{nor_upper if nor_upper != float('inf') else 'max'})"
                count.rename(columns={time_category_column: 'count_in_threshold_range'}, inplace=True)

                results.append(count)
    
    # Combine all results
    if results:
        return pd.concat(results)
    else:
        return pd.DataFrame()

In [ ]:
thresholds_nimo = [0, 1, 90, 180, 360]
thresholds_nor = [0, 1, 2000, 4000, 6000, 8000, 10000, 15000, 20000, 30000, 40000, 50000]

counts_df = count_days_in_threshold_ranges(nimo_nor_daily_df, thresholds_nimo, thresholds_nor)


In [ ]:
thresholds_nimo = [0, 15, 30, 60, 120]
thresholds_nor = [0, 1, 1*4*60, 5*4*60, 10*4*60, 15*4*60, 20*4*60, 30*4*60]

counts_4h_df = count_days_in_threshold_ranges(nimo_nor_4h_cat_df, thresholds_nimo, thresholds_nor, time_category_column='relative_start_4h_cat')

In [ ]:
# for every combination of nimo_threshold and nor_threshold, calculate the correlation
def event_count_to_mrs_correlation(events_df, event_count_col='count_in_threshold_range', outcome_col='mrs_1y'):
    """
    Compute the Pearson correlation coefficient between event counts and mRS_1y for each nimodipine and noradrenaline threshold combination.
    
    Arguments:
        - events_df: DataFrame with event counts
    
    Returns: correlation_df
    """
    correlation_results = []

    for (nimodipine_threshold, noradrenaline_threshold), group in events_df.groupby(['nimodipine_lower', 'noradrenaline_lower']):
        temp_df = group.copy()
        temp_df[outcome_col] = pd.to_numeric(temp_df[outcome_col], errors='coerce')
        temp_df.dropna(subset=[event_count_col, outcome_col], inplace=True)
        if temp_df.empty:
            continue
        if len(temp_df) > 1:  # Ensure there are enough data points to compute correlation
            corr, p_value = pearsonr(temp_df[event_count_col], temp_df[outcome_col])
            correlation_results.append({
                'nimodipine_threshold': nimodipine_threshold,
                'noradrenaline_threshold': noradrenaline_threshold,
                'correlation_coefficient': corr,
                'p_value': p_value
            })
    
    return pd.DataFrame(correlation_results)

In [ ]:
correlation_df = event_count_to_mrs_correlation(counts_df, event_count_col='count_in_threshold_range', outcome_col='mRS_FU_1y')

In [ ]:
corr_4h_cat_df = event_count_to_mrs_correlation(counts_4h_df, event_count_col='count_in_threshold_range', outcome_col='mRS_FU_1y')

In [ ]:
corr_4h_cat_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(correlation_df.pivot_table(
        index='noradrenaline_threshold',
        columns='nimodipine_threshold',
        values='correlation_coefficient'
    ).reindex(index=sorted(correlation_df['noradrenaline_threshold'].unique(), reverse=True)),
        annot=True, cmap='seismic', center=0, ax=ax)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_4h_cat_df.pivot_table(
        index='noradrenaline_threshold',
        columns='nimodipine_threshold',
        values='correlation_coefficient'
    ).reindex(index=sorted(corr_4h_cat_df['noradrenaline_threshold'].unique(), reverse=True)),
        annot=True, cmap='seismic', center=0, ax=ax)
plt.show()

In [ ]:
import statsmodels.api as sm

def event_count_to_DCI_coefficient(events_df, event_count_column='event_count', DCI_column='DCI_ischemia'):
    """
    Compute the logistic regression coefficient between event counts and DCI_ischemia for each intensity and duration threshold.
    Arguments:
        - events_df: DataFrame with event counts
        - DCI_column: column name for DCI_ischemia (default is 'DCI_ischemia')
    Returns: coefficient_df
    """
    coefficient_results = []
    
    for (nimodipine_threshold, noradrenaline_threshold), group in events_df.groupby(['nimodipine_lower', 'noradrenaline_lower']):
        temp_df = group.copy()
        temp_df.dropna(subset=[event_count_column, DCI_column], inplace=True)

        # Skip if too few data points
        if len(temp_df) <= 1:
            continue
            
        # Check for zero variance
        if temp_df[event_count_column].std() == 0:
            continue
            
        try:
            log_model = sm.Logit(temp_df[DCI_column],
                                 sm.add_constant(temp_df[event_count_column]))
            log_model_result = log_model.fit(disp=0, method='bfgs')  # Try different solver
            coefficient_results.append({
                'nimodipine_threshold': nimodipine_threshold,
                'noradrenaline_threshold': noradrenaline_threshold,
                'coefficient': log_model_result.params[event_count_column],
                'p_value': log_model_result.pvalues[event_count_column],
                'n_samples': len(temp_df)
            })
        except Exception as e:
            print(f"Error at threshold {nimodipine_threshold}/{noradrenaline_threshold}: {e}")

    
    return pd.DataFrame(coefficient_results)

In [ ]:
dci_coef_4h_cat_df = event_count_to_DCI_coefficient(counts_4h_df, event_count_column='count_in_threshold_range', DCI_column='DCI_ischemia')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(dci_coef_4h_cat_df.pivot_table(
        index='noradrenaline_threshold',
        columns='nimodipine_threshold',
        values='coefficient'
    ).reindex(index=sorted(dci_coef_4h_cat_df['noradrenaline_threshold'].unique(), reverse=True)),
        annot=True, cmap='seismic', center=0, ax=ax)
plt.show()

In [ ]:
counts_4h_df

among patientes with nor > threshold
- what impact does reducing nimo have? 

In [ ]:
nimo_nor_4h_cat_df

In [ ]:
nimo_nor_4h_cat_df['Rate_nor'] = nimo_nor_4h_cat_df['Menge_nor'] / (4 * 60)

In [ ]:
threshold_nor_rate = 15
temp_df = nimo_nor_4h_cat_df[nimo_nor_4h_cat_df['Rate_nor'] >= threshold_nor_rate]

import os
os.environ["R_HOME"] = "/Library/Frameworks/R.framework/Versions/4.1/Resources"
from pymer4.models import Lmer

temp_df.dropna(subset=['Menge_nimo', 'mRS_FU_1y'], inplace=True)

# Fit a linear mixed-effects model
model = Lmer("mRS_FU_1y ~ Menge_nimo + (1|pNr)", data=temp_df, family="gaussian")
model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
print(model.summary())

In [ ]:
temp_df.Menge_nimo.hist(bins=50)